# Design rules

In [11]:
# ---- Import capacitance model and process constants ----
from design_rules import CapacitanceModel, Geometry, DEFAULT_90NM, F16_EXAM, ureg

# Units
um = ureg.um
nm = ureg.nm
aF = ureg.aF
fF = ureg.fF
MHz = ureg.MHz
GHz = ureg.GHz
V = ureg.V
nW = ureg.nW


## Geometry inputs

Most of the exersizes works with minimal geometries. Fx. (W,L)=(200nm,100nm). 
- W = width of the channel (active diffusion) under the gate
- L = channel length, i.e. the distance from source to drain under the gate

In [12]:
W = 200 * nm   # um (200 nm)
L = 100 * nm   # um (100 nm)

## Calculating capacitance from a stick diagram

...

An example (NAND3) of diffusion regions can be seen below.

<img src="img/nand3-diffusions-fig4.17.png" width="25%">

In [13]:
geometry = Geometry(
    w = W,
    l = L,

    # ---- Diffusion lengths ----
    diff_con = 260 * nm,  # shared contacted diffusion
    diff_end = 230 * nm,  # isolated contacted diffusion
    diff_unc = 150 * nm,  # merged uncontacted diffusion
)

In [14]:
# Build a model for the default process
model = CapacitanceModel(constants=DEFAULT_90NM, geometry=geometry)

# E02205 F16 Problem 2

The following is from an older exam that is using different constants that the 90nm process used in class.

In [15]:
# Use the exam process constants for the following calculations
model = CapacitanceModel(constants=F16_EXAM, geometry=geometry)

terms = model.capacitance_terms()
g = model.geometry
term_specs = [
    ("c_p_con", g.diff_con, terms.c_p_con),
    ("c_p_end", g.diff_end, terms.c_p_end),
    ("c_p_unc", g.diff_unc, terms.c_p_unc),
    ("c_n_con", g.diff_con, terms.c_n_con),
    ("c_n_end", g.diff_end, terms.c_n_end),
    ("c_n_unc", g.diff_unc, terms.c_n_unc),
]

# Display the table as md for the notebook
md_table = "| term      | w (nm) | l (nm) | A=w*l (nm^2) | P=2(w+l) (nm) | C_total (aF) |\n"
md_table += "|-----------|--------|--------|--------------|---------------|----------------|\n"
for name, l_seg, total in term_specs:
    A = g.w * l_seg
    P = 2.0 * (g.w + l_seg)
    md_table += f"| {name:10s} | ${g.w.to(nm):~L}$ | ${l_seg.to(nm):~L}$ | ${A.to(nm**2):~L}$ | ${P.to(nm):~L}$ | ${total.to(aF):.0f~L}$ |\n"

from IPython.display import Markdown, display

display(Markdown(md_table))
print(md_table)


| term      | w (nm) | l (nm) | A=w*l (nm^2) | P=2(w+l) (nm) | C_total (aF) |
|-----------|--------|--------|--------------|---------------|----------------|
| c_p_con    | $200\ \mathrm{nm}$ | $260\ \mathrm{nm}$ | $52000\ \mathrm{nm}^{2}$ | $920.0\ \mathrm{nm}$ | $104\ \mathrm{aF}$ |
| c_p_end    | $200\ \mathrm{nm}$ | $230\ \mathrm{nm}$ | $46000\ \mathrm{nm}^{2}$ | $860.0\ \mathrm{nm}$ | $93\ \mathrm{aF}$ |
| c_p_unc    | $200\ \mathrm{nm}$ | $150\ \mathrm{nm}$ | $30000\ \mathrm{nm}^{2}$ | $700.0\ \mathrm{nm}$ | $65\ \mathrm{aF}$ |
| c_n_con    | $200\ \mathrm{nm}$ | $260\ \mathrm{nm}$ | $52000\ \mathrm{nm}^{2}$ | $920.0\ \mathrm{nm}$ | $104\ \mathrm{aF}$ |
| c_n_end    | $200\ \mathrm{nm}$ | $230\ \mathrm{nm}$ | $46000\ \mathrm{nm}^{2}$ | $860.0\ \mathrm{nm}$ | $93\ \mathrm{aF}$ |
| c_n_unc    | $200\ \mathrm{nm}$ | $150\ \mathrm{nm}$ | $30000\ \mathrm{nm}^{2}$ | $700.0\ \mathrm{nm}$ | $64\ \mathrm{aF}$ |


| term      | w (nm) | l (nm) | A=w*l (nm^2) | P=2(w+l) (nm) | C_total (aF) |
|-----------|--------|--------|--------------|---------------|----------------|
| c_p_con    | $200\ \mathrm{nm}$ | $260\ \mathrm{nm}$ | $52000\ \mathrm{nm}^{2}$ | $920.0\ \mathrm{nm}$ | $104\ \mathrm{aF}$ |
| c_p_end    | $200\ \mathrm{nm}$ | $230\ \mathrm{nm}$ | $46000\ \mathrm{nm}^{2}$ | $860.0\ \mathrm{nm}$ | $93\ \mathrm{aF}$ |
| c_p_unc    | $200\ \mathrm{nm}$ | $150\ \mathrm{nm}$ | $30000\ \mathrm{nm}^{2}$ | $700.0\ \mathrm{nm}$ | $65\ \mathrm{aF}$ |
| c_n_con    | $200\ \mathrm{nm}$ | $260\ \mathrm{nm}$ | $52000\ \mathrm{nm}^{2}$ | $920.0\ \mathrm{nm}$ | $104\ \mathrm{aF}$ |
| c_n_end    | $200\ \mathrm{nm}$ | $230\ \mathrm{nm}$ | $46000\ \mathrm{nm}^{2}$ | $860.0\ \mathrm{nm}$ | $93\ \mathrm{aF}$ |
| c_n_unc    | $200\ \mathrm{nm}$ | $150\ \mathrm{nm}$ | $30000\ \mathrm{nm}^{2}$ | $700.0\ \mathrm{nm}$ | $64\ \mathrm{aF}$ |



<img src="img/dff-td.png" width="50%">
<img src="img/dff-stick.png" width="50%">

In [16]:
# Short aliases
c_p_con = terms.c_p_con
c_p_end = terms.c_p_end
c_p_unc = terms.c_p_unc
c_n_con = terms.c_n_con
c_n_end = terms.c_n_end
c_n_unc = terms.c_n_unc

c_g = terms.c_gate

C_load = 2.0e3 * aF  # aF = 2 fF external load

C_q1 = c_p_end + c_p_con
C_q2 = c_n_unc
C_q3 = c_p_unc
C_x  = c_n_end + c_p_end + 2*c_g
C_f  = c_n_end + c_p_end + C_load


In [17]:
# Final results table
result_rows = [
    ("C_{q1}", C_q1),
    ("C_{q2}", C_q2),
    ("C_{q3}", C_q3),
    ("C_{x}", C_x),
    ("C_{f}", C_f),
]

# Print the results as md for the notebook
md_result_table = "| Result | C (aF) | C (fF) |\n"
md_result_table += "|--------|---------|---------|\n"
for name, value in result_rows:
    md_result_table += f"| ${name:6s}$ | ${value.to(aF):.3f~L}$ | ${value.to(fF):.3f~L}$ |\n"

display(Markdown(md_result_table))
print(md_result_table)

| Result | C (aF) | C (fF) |
|--------|---------|---------|
| $C_{q1}$ | $196.676\ \mathrm{aF}$ | $0.197\ \mathrm{fF}$ |
| $C_{q2}$ | $64.130\ \mathrm{aF}$ | $0.064\ \mathrm{fF}$ |
| $C_{q3}$ | $64.860\ \mathrm{aF}$ | $0.065\ \mathrm{fF}$ |
| $C_{x} $ | $1086.478\ \mathrm{aF}$ | $1.086\ \mathrm{fF}$ |
| $C_{f} $ | $2186.478\ \mathrm{aF}$ | $2.186\ \mathrm{fF}$ |


| Result | C (aF) | C (fF) |
|--------|---------|---------|
| $C_{q1}$ | $196.676\ \mathrm{aF}$ | $0.197\ \mathrm{fF}$ |
| $C_{q2}$ | $64.130\ \mathrm{aF}$ | $0.064\ \mathrm{fF}$ |
| $C_{q3}$ | $64.860\ \mathrm{aF}$ | $0.065\ \mathrm{fF}$ |
| $C_{x} $ | $1086.478\ \mathrm{aF}$ | $1.086\ \mathrm{fF}$ |
| $C_{f} $ | $2186.478\ \mathrm{aF}$ | $2.186\ \mathrm{fF}$ |



# Estimate the dynamic power consumption assuming that, on the average, every input combination occurs equally frequently.

We estimate the dynamic power consumption using the formula:

$$P_d = \sum{(A_i \cdot C_i)} \cdot V_{dd}^2 \cdot f$$


Activity is is found by conting the number of input combinations that causes a discharge of the node. 

$q_1$ is discharged (through the pull-down network) for 1 out of 4 input combinations, when $a = 1$ and $b = 1$.

$$A_{q1} = \frac{1}{4}$$

$q_2$ is discharged for 2 out of 4 input combinations, when $b = 1$ ($a$ can be whatever).

$$A_{q2} = \frac{2}{4} = \frac{1}{2}$$

$x$ is discharged (through the pull-down network) for 1 out of 4 input combinations, when $a = 1$ and $b = 1$.

$$A_x = \frac{1}{4}$$

$q_3$ ans $f$ is discharged the same as x.

$$A_{\ell} = \frac{1}{4}$$


In [18]:
P_dx = ((C_q1 / 4) + (C_q2 / 2) + (C_x / 4)) * ((1.0 * V) ** 2) * 650 * MHz # fF * V * GHz = nW
print(f"Dynamic power P_dx = {P_dx.to(nW):.3f~P} nW")

P_df = (C_q3 + C_f) / 4 * ((1.0 * V) ** 2) * 650 * MHz # fF * V * GHz = nW
print(f"Dynamic power P_df = {P_df.to(nW):.3f~P} nW")

P_d = P_dx + P_df
print(f"Total dynamic power P_d = {P_d.to(nW):.3f~P} nW")

Dynamic power P_dx = 229.355 nW nW
Dynamic power P_df = 365.842 nW nW
Total dynamic power P_d = 595.197 nW nW


In [19]:
C_eff = ((C_q1 / 4) + (C_q2 / 2) + (C_x / 4))
Vdd = 1.0 * V
f = 650 * MHz

P = C_eff.to(fF) * Vdd**2 * f
print(f"{P.to(ureg.nW):.3f~P}")

229.355 nW


# RC diagrams

## Questions

- How do I draw the RC diagram for a given stick diagram?
- How do I recognize what transition is the worst case for a given diagram?



The core formula is:

$$t_{pd} \approx \ln(2) \cdot \sum_i C_i \cdot R_{\text{shared},i}$$

where:

- $C_i$: capacitance at node $i$
- $R_{\text{shared},i}$: resistance common to both:
- path from source → output
- path from source → node $i$
